# Docling Spike — PDF Extraction Quality Evaluation

**Branch**: `012-docling-spike`  
**Goal**: Compare Docling vs our current pymupdf4llm pipeline across three dimensions:
- **Extraction quality** — noise, encoding, structure fidelity
- **Chunk quality** — size distribution, stubs, giants, heading context
- **Per-component decisions** — keep ours vs adopt Docling's

## Experiment design

| Run | Extraction | Chunking | Label |
|-----|-----------|---------|-------|
| A (baseline, 011) | pymupdf4llm + CorpusCleaner | AgenticChunker | already in ChromaDB |
| B | Docling → Markdown | HeadingChunker (AgenticChunker proxy) | this notebook |
| C | Docling → DoclingDocument | Docling HybridChunker | this notebook |

A vs B → **is Docling extraction better than pymupdf4llm + our cleaning rules?**  
B vs C → **is Docling's native chunker better than ours?**

---
Run from repo root:
```
jupyter notebook notebooks/docling_spike.ipynb
```

## 0. Installation

Docling is **not** in `packages/rag/pyproject.toml` yet — install it directly for this spike.

In [ ]:
# Run once — Docling pulls in PyTorch + ML models (~1-2 GB on first run)
# Model download happens on first DocumentConverter() call, not here
!pip install docling --quiet

In [ ]:
import sys, os, re, json, textwrap
from pathlib import Path
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
for pkg in ['packages/rag', 'packages/llm', 'packages/core', 'packages/storage']:
    p = os.path.join(ROOT, pkg)
    if p not in sys.path:
        sys.path.insert(0, p)

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_rows', 60)

print('Python:', sys.version)
print('Root:', ROOT)

## 1. Configuration — set your PDF path here

In [ ]:
# ── EDIT THIS ──────────────────────────────────────────────────────────────────
PDF_PATH = os.environ.get('SPIKE_PDF_PATH', '/path/to/ED4_Players_Guide.pdf')
# ───────────────────────────────────────────────────────────────────────────────

# Pages to deep-dive (0-indexed): pick a normal text page, a table page,
# a multi-column page, a stat-block page, and a TOC/index page.
SAMPLE_PAGES = [10, 50, 100, 200, 400]  # adjust to pages that exist in your PDF

pdf_path = Path(PDF_PATH)
if not pdf_path.exists():
    print(f'⚠️  PDF not found at: {PDF_PATH}')
    print('Set PDF_PATH above or export SPIKE_PDF_PATH=/your/path/to/file.pdf')
else:
    print(f'✓ PDF found: {pdf_path.name} ({pdf_path.stat().st_size / 1_048_576:.1f} MB)')

## 2. Baseline extraction — pymupdf4llm (Run A equivalent)

Extract the full document with our current pipeline. No cleaning applied here —
we want to measure **raw extractor noise** before any CorpusCleaner rules.

In [ ]:
import pymupdf4llm
import fitz

print('Extracting with pymupdf4llm...')
pymupdf_md = pymupdf4llm.to_markdown(str(pdf_path))

# Also extract page-by-page for per-page analysis
doc = fitz.open(str(pdf_path))
pymupdf_pages = []
for i in range(len(doc)):
    page_md = pymupdf4llm.to_markdown(str(pdf_path), pages=[i])
    pymupdf_pages.append(page_md)
doc.close()

print(f'Total chars  : {len(pymupdf_md):,}')
print(f'Pages        : {len(pymupdf_pages)}')
print(f'Avg chars/page: {len(pymupdf_md) // len(pymupdf_pages):,}')

## 3. Docling extraction — DocumentConverter

First run will download ML models (layout, table structure). Subsequent runs use cache.

In [ ]:
from docling.document_converter import DocumentConverter

print('Converting with Docling (first run downloads models)...')
converter = DocumentConverter()
result = converter.convert(str(pdf_path))
docling_doc = result.document

print(f'Conversion status: {result.status}')
print(f'Pages processed  : {len(docling_doc.pages)}')

In [ ]:
# Inspect the DoclingDocument structure
element_types = Counter()
for item, _ in docling_doc.iterate_items():
    element_types[type(item).__name__] += 1

print('Document element types:')
for etype, count in element_types.most_common():
    print(f'  {etype:<30} {count:>5}')

In [ ]:
# Export to Markdown — this is what Run B will use
docling_md = docling_doc.export_to_markdown()

print(f'Docling Markdown chars: {len(docling_md):,}')
print(f'pymupdf4llm chars     : {len(pymupdf_md):,}')
print(f'Ratio (docling/pymupdf): {len(docling_md)/len(pymupdf_md):.2f}')

## 4. Noise audit — FR-001..006 checks on both extractors

For each noise type we caught in spec 011, measure whether it appears in
pymupdf4llm raw output vs Docling Markdown output. A lower count in Docling
means that cleaning rule can be retired.

In [ ]:
# Noise detection patterns (same as CorpusCleaner)
_WIN1252_MOJIBAKE   = re.compile(r'[\x80-\x9f\ufffd]')
_DROPCAP_GAP        = re.compile(r'(?m)^([A-Z])\n([a-z])')
_IMAGE_PLACEHOLDER  = re.compile(r'==>[ \t]*[^\n]*?<==|-{3,}[ \t]*Start of picture text', re.IGNORECASE)
_PAGE_NUMBER        = re.compile(r'(?m)^[ \t]*\d{1,4}[ \t]*$')
_TOC_DOT_LEADER     = re.compile(r'\.{5,}')
_HYPHEN_BREAK       = re.compile(r'\w+-\n\w')

def count_noise(text: str) -> dict:
    return {
        'encoding_mojibake':   len(_WIN1252_MOJIBAKE.findall(text)),
        'dropcap_gaps':        len(_DROPCAP_GAP.findall(text)),
        'image_placeholders':  len(_IMAGE_PLACEHOLDER.findall(text)),
        'stranded_page_nums':  len(_PAGE_NUMBER.findall(text)),
        'toc_dot_leaders':     len(_TOC_DOT_LEADER.findall(text)),
        'hyphen_linebreaks':   len(_HYPHEN_BREAK.findall(text)),
    }

noise_pymupdf  = count_noise(pymupdf_md)
noise_docling  = count_noise(docling_md)

df_noise = pd.DataFrame({
    'pymupdf4llm (raw)': noise_pymupdf,
    'Docling (raw)':     noise_docling,
})
df_noise['delta'] = df_noise['Docling (raw)'] - df_noise['pymupdf4llm (raw)']
df_noise['verdict'] = df_noise['delta'].apply(
    lambda d: '✓ Docling better' if d < 0 else ('= same' if d == 0 else '✗ Docling worse')
)

print('Noise audit — raw extractor output (before any cleaning):')
display(df_noise)

In [ ]:
# Check whether Docling separates furniture (headers/footers) from body text
# This tells us if frontmatter/page-number patterns are gone without explicit rules

from docling_core.types.doc import DocItemLabel

furniture_texts = []
body_texts = []

for item, level in docling_doc.iterate_items():
    if hasattr(item, 'text'):
        label = getattr(item, 'label', None)
        if label in (DocItemLabel.PAGE_HEADER, DocItemLabel.PAGE_FOOTER):
            furniture_texts.append(item.text)
        else:
            body_texts.append(item.text)

print(f'Furniture elements (headers/footers): {len(furniture_texts)}')
print(f'Body elements: {len(body_texts)}')
if furniture_texts:
    print('\nSample furniture text (first 10):')
    for t in furniture_texts[:10]:
        print(f'  {repr(t[:80])}')

In [ ]:
# Table extraction quality — how many tables did Docling find?
# pymupdf4llm converts tables to Markdown pipe-tables; Docling uses TableItem with cell structure

from docling_core.types.doc import TableItem

tables = [item for item, _ in docling_doc.iterate_items() if isinstance(item, TableItem)]

print(f'Tables found by Docling: {len(tables)}')

# Count pipe-table rows in pymupdf4llm output
_TABLE_ROW = re.compile(r'^\|.+\|', re.MULTILINE)
pymupdf_table_rows = len(_TABLE_ROW.findall(pymupdf_md))
print(f'Pipe-table rows in pymupdf4llm output: {pymupdf_table_rows}')

# Show first 3 tables from Docling
print('\nFirst 3 Docling tables (exported as Markdown):')
for i, tbl in enumerate(tables[:3]):
    print(f'\n--- Table {i+1} (page {tbl.prov[0].page_no if tbl.prov else "?"}) ---')
    try:
        print(tbl.export_to_markdown())
    except Exception as e:
        print(f'  (export failed: {e})')

## 5. Side-by-side page comparison

For each sample page, show both extractors' output. Focus on:
- Multi-column layouts (reading order)
- Stat blocks (table-like dense structure)
- Pages with noise (headers/footers, page numbers)

In [ ]:
def show_page_comparison(page_num: int, chars: int = 800):
    """Show pymupdf4llm vs Docling output for a given 0-indexed page number."""
    print(f'{'='*80}')
    print(f'PAGE {page_num + 1} (0-indexed: {page_num})')
    print(f'{'='*80}')

    # pymupdf4llm
    pymu_text = pymupdf_pages[page_num] if page_num < len(pymupdf_pages) else '(out of range)'
    print(f'\n--- pymupdf4llm ({len(pymu_text)} chars) ---')
    print(textwrap.indent(pymu_text[:chars], '  '))
    if len(pymu_text) > chars:
        print(f'  ... [{len(pymu_text) - chars} more chars]')

    # Docling — find elements on this page (1-indexed in Docling)
    docling_page_items = []
    for item, _ in docling_doc.iterate_items():
        if hasattr(item, 'prov') and item.prov:
            for prov in item.prov:
                if prov.page_no == page_num + 1:
                    if hasattr(item, 'text'):
                        docling_page_items.append(('text', item.label.value if hasattr(item, 'label') else '?', item.text))
                    elif isinstance(item, TableItem):
                        try:
                            docling_page_items.append(('table', 'table', item.export_to_markdown()))
                        except:
                            docling_page_items.append(('table', 'table', '(export failed)'))
                    break

    docling_text = '\n'.join(f'[{label}] {text}' for _, label, text in docling_page_items)
    print(f'\n--- Docling ({len(docling_text)} chars, {len(docling_page_items)} elements) ---')
    print(textwrap.indent(docling_text[:chars], '  '))
    if len(docling_text) > chars:
        print(f'  ... [{len(docling_text) - chars} more chars]')

for pg in SAMPLE_PAGES:
    show_page_comparison(pg)
    print()

## 6. Chunk quality — Run B (Docling → HeadingChunker)

HeadingChunker is used as a proxy for AgenticChunker (which requires Ollama).
The harness run with the full AgenticChunker is done separately.  
This gives us chunk size distribution and stub/giant counts.

In [ ]:
from rag.knowledge.chunker import HeadingChunker

chunker_heading = HeadingChunker()

# Run B: Docling Markdown → HeadingChunker
run_b_chunks = chunker_heading.chunk(docling_md)

# Baseline: pymupdf4llm Markdown → HeadingChunker (apples-to-apples, no cleaning)
run_a_proxy_chunks = chunker_heading.chunk(pymupdf_md)

print(f'Run A proxy (pymupdf → HeadingChunker): {len(run_a_proxy_chunks)} chunks')
print(f'Run B       (Docling → HeadingChunker) : {len(run_b_chunks)} chunks')

## 7. Chunk quality — Run C (Docling → HybridChunker)

Uses Docling's native `HybridChunker` on the `DoclingDocument` object directly.
Each chunk carries heading/caption metadata — this is the key advantage over Markdown-based chunking.

In [ ]:
try:
    from docling.chunking import HybridChunker
    hybrid_chunker = HybridChunker()
    run_c_chunk_objects = list(hybrid_chunker.chunk(docling_doc))
    run_c_chunks = [c.text for c in run_c_chunk_objects]
    print(f'Run C (Docling → HybridChunker): {len(run_c_chunks)} chunks')
    _hybrid_available = True
except Exception as e:
    print(f'HybridChunker unavailable: {e}')
    print('Run C will be skipped in comparisons below.')
    run_c_chunks = []
    run_c_chunk_objects = []
    _hybrid_available = False

In [ ]:
# Inspect Docling chunk metadata — headings, captions, provenance
# This replaces what our BreadcrumbExtractor does
if _hybrid_available and run_c_chunk_objects:
    print('Sample chunk metadata from Docling HybridChunker (first 5 chunks):')
    for i, chunk in enumerate(run_c_chunk_objects[:5]):
        print(f'\n--- Chunk {i+1} ---')
        print(f'  text preview  : {chunk.text[:120]!r}')
        meta = chunk.meta
        print(f'  meta type     : {type(meta).__name__}')
        # Try to extract heading path
        headings = getattr(meta, 'headings', None) or getattr(meta, 'doc_items', [])
        print(f'  headings      : {headings}')
        captions = getattr(meta, 'captions', None)
        if captions:
            print(f'  captions      : {captions}')
        # Show all meta attributes
        print(f'  meta attrs    : {[a for a in dir(meta) if not a.startswith("_")]}')

## 8. Chunk quality metrics — comparison table

In [ ]:
MIN_CHARS = 150
MAX_CHARS = 15000

def chunk_stats(chunks: list[str], label: str) -> dict:
    if not chunks:
        return {'run': label, 'count': 0}
    lengths = [len(c) for c in chunks]
    return {
        'run':           label,
        'count':         len(chunks),
        'stubs (<150)':  sum(1 for l in lengths if l < MIN_CHARS),
        'giants (>15k)': sum(1 for l in lengths if l > MAX_CHARS),
        'avg_chars':     int(sum(lengths) / len(lengths)),
        'median_chars':  int(sorted(lengths)[len(lengths)//2]),
        'min_chars':     min(lengths),
        'max_chars':     max(lengths),
        'p90_chars':     int(sorted(lengths)[int(len(lengths)*0.9)]),
    }

stats = [
    chunk_stats(run_a_proxy_chunks, 'A-proxy (pymupdf → Heading)'),
    chunk_stats(run_b_chunks,       'B       (Docling → Heading)'),
]
if _hybrid_available:
    stats.append(chunk_stats(run_c_chunks, 'C       (Docling → Hybrid)'))

df_stats = pd.DataFrame(stats).set_index('run')
print('Chunk quality metrics:')
display(df_stats)

In [ ]:
# Length distribution histograms
runs_to_plot = [
    ('A-proxy (pymupdf)', run_a_proxy_chunks),
    ('B (Docling → Heading)', run_b_chunks),
]
if _hybrid_available:
    runs_to_plot.append(('C (Docling → Hybrid)', run_c_chunks))

fig, axes = plt.subplots(1, len(runs_to_plot), figsize=(6 * len(runs_to_plot), 4), sharey=True)
if len(runs_to_plot) == 1:
    axes = [axes]

for ax, (label, chunks) in zip(axes, runs_to_plot):
    lengths = [min(len(c), 5000) for c in chunks]
    ax.hist(lengths, bins=50, color='steelblue', alpha=0.8)
    ax.axvline(MIN_CHARS, color='red', linestyle='--', label=f'min ({MIN_CHARS})')
    ax.axvline(min(MAX_CHARS, 5000), color='orange', linestyle='--', label=f'max ({MAX_CHARS})')
    ax.set_title(label)
    ax.set_xlabel('chunk length (chars, clipped at 5000)')
    ax.legend(fontsize=8)

axes[0].set_ylabel('chunk count')
plt.suptitle('Chunk length distributions', y=1.02)
plt.tight_layout()
plt.show()

## 9. Heading/breadcrumb coverage

How well does each approach provide heading context per chunk?
This replaces what `BreadcrumbExtractor` was doing.

In [ ]:
_HEADING_RE = re.compile(r'^#{1,3} .+', re.MULTILINE)

def heading_coverage(chunks: list[str]) -> dict:
    """Fraction of chunks that start with or contain a heading."""
    starts_with = sum(1 for c in chunks if c.strip().startswith('#'))
    contains    = sum(1 for c in chunks if _HEADING_RE.search(c))
    return {
        'starts_with_heading': f'{starts_with}/{len(chunks)} ({starts_with/len(chunks)*100:.1f}%)',
        'contains_heading':    f'{contains}/{len(chunks)} ({contains/len(chunks)*100:.1f}%)',
        'orphaned (no heading)': f'{len(chunks)-contains}/{len(chunks)} ({(len(chunks)-contains)/len(chunks)*100:.1f}%)',
    }

print('Heading coverage in Run B (Docling → Heading):')
for k, v in heading_coverage(run_b_chunks).items():
    print(f'  {k:<30} {v}')

if _hybrid_available and run_c_chunks:
    print('\nHeading coverage in Run C (Docling → Hybrid):')
    for k, v in heading_coverage(run_c_chunks).items():
        print(f'  {k:<30} {v}')
    
    # Also show how many Hybrid chunks have structured metadata headings
    chunks_with_meta_headings = sum(
        1 for c in run_c_chunk_objects
        if getattr(c.meta, 'headings', None)
    )
    print(f'\n  Chunks with Docling native heading metadata: {chunks_with_meta_headings}/{len(run_c_chunk_objects)}')

## 10. Game-specific structure — stat block detection

Docling won't know about TTRPG stat blocks. Check whether they survive cleanly
in Docling's output (kept as tables or prose) vs our reconstruction logic.

In [ ]:
# Heuristic: stat blocks contain patterns like "DEX: 12" or "Initiative:" or "Wound Threshold:"
_STAT_PATTERNS = re.compile(
    r'(?:DEX|STR|TOU|PER|WIL|CHA|Initiative|Wound Threshold|Death Rating|Knockdown|Unconscious)\s*:',
    re.IGNORECASE
)

pymupdf_stat_hits = _STAT_PATTERNS.findall(pymupdf_md)
docling_stat_hits = _STAT_PATTERNS.findall(docling_md)

print(f'Stat block attribute mentions in pymupdf4llm: {len(pymupdf_stat_hits)}')
print(f'Stat block attribute mentions in Docling    : {len(docling_stat_hits)}')

# Find a stat block in the Docling output and print the surrounding text
for m in _STAT_PATTERNS.finditer(docling_md):
    start = max(0, m.start() - 200)
    end = min(len(docling_md), m.end() + 400)
    print(f'\nSample Docling stat block context (pos {m.start()}):')
    print(docling_md[start:end])
    print('---')
    break

## 11. Performance — conversion time

Docling is heavier than pymupdf4llm. Measure wall-clock time per page.

In [ ]:
import time

# Time pymupdf4llm on a sample of pages
TIMING_PAGES = min(20, len(pymupdf_pages))

t0 = time.perf_counter()
for i in range(TIMING_PAGES):
    pymupdf4llm.to_markdown(str(pdf_path), pages=[i])
pymupdf_time = time.perf_counter() - t0

# Docling already ran — estimate from full conversion time
# (re-run on a small slice for a fair comparison)
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions

pipeline_opts = PdfPipelineOptions()
pipeline_opts.do_ocr = False
sample_converter = DocumentConverter(
    format_options={"pdf": PdfFormatOption(pipeline_options=pipeline_opts)}
)

t0 = time.perf_counter()
sample_result = sample_converter.convert(str(pdf_path), max_num_pages=TIMING_PAGES)
docling_time = time.perf_counter() - t0

print(f'Timing over {TIMING_PAGES} pages:')
print(f'  pymupdf4llm : {pymupdf_time:.2f}s  ({pymupdf_time/TIMING_PAGES:.3f}s/page)')
print(f'  Docling     : {docling_time:.2f}s  ({docling_time/TIMING_PAGES:.3f}s/page)')
print(f'  Ratio       : {docling_time/pymupdf_time:.1f}x slower')

full_pages = len(pymupdf_pages)
projected_docling = docling_time / TIMING_PAGES * full_pages
print(f'\nProjected full-doc ({full_pages} pages):')
print(f'  pymupdf4llm : ~{pymupdf_time/TIMING_PAGES*full_pages/60:.1f} min')
print(f'  Docling     : ~{projected_docling/60:.1f} min')

## 12. Summary — per-component decision table

Fill in the **Decision** column after reviewing the cells above.

In [ ]:
# Auto-populate what we can measure; manually fill the rest
stub_b = sum(1 for c in run_b_chunks if len(c) < MIN_CHARS)
stub_a = sum(1 for c in run_a_proxy_chunks if len(c) < MIN_CHARS)
stub_c = sum(1 for c in run_c_chunks if len(c) < MIN_CHARS) if _hybrid_available else 'n/a'

noise_delta_enc = noise_docling['encoding_mojibake'] - noise_pymupdf['encoding_mojibake']
noise_delta_img = noise_docling['image_placeholders'] - noise_pymupdf['image_placeholders']
noise_delta_pn  = noise_docling['stranded_page_nums'] - noise_pymupdf['stranded_page_nums']

summary_rows = [
    {
        'Component': 'Extraction (pymupdf4llm vs Docling)',
        'Metric': f'encoding noise: {noise_pymupdf["encoding_mojibake"]} → {noise_docling["encoding_mojibake"]} | page# noise: {noise_pymupdf["stranded_page_nums"]} → {noise_docling["stranded_page_nums"]}',
        'Decision': '← fill in after review',
    },
    {
        'Component': 'Image placeholder cleaning (FR-003)',
        'Metric': f'placeholders: {noise_pymupdf["image_placeholders"]} → {noise_docling["image_placeholders"]}',
        'Decision': '← fill in after review',
    },
    {
        'Component': 'Frontmatter / furniture separation',
        'Metric': f'furniture elements: {len(furniture_texts)}',
        'Decision': '← fill in after review',
    },
    {
        'Component': 'Table extraction',
        'Metric': f'Docling tables: {len(tables)} | pymupdf pipe-rows: {pymupdf_table_rows}',
        'Decision': '← fill in after review',
    },
    {
        'Component': 'Chunking (HeadingChunker proxy)',
        'Metric': f'stubs A-proxy: {stub_a} | B: {stub_b} | C: {stub_c}',
        'Decision': '← fill in after review',
    },
    {
        'Component': 'Breadcrumb / heading context',
        'Metric': 'see Section 9',
        'Decision': '← fill in after review',
    },
    {
        'Component': 'Stat block reconstruction',
        'Metric': f'stat hits pymupdf: {len(pymupdf_stat_hits)} | Docling: {len(docling_stat_hits)}',
        'Decision': 'keep CorpusCleaner stat block rules regardless',
    },
    {
        'Component': 'Performance',
        'Metric': f'see Section 11',
        'Decision': '← fill in after review',
    },
]

df_summary = pd.DataFrame(summary_rows)
print('=== SPIKE DECISION TABLE ===')
display(df_summary)

## 13. Findings & next steps

*(Fill this in after running the notebook)*

### Extraction layer
- [ ] Docling reduces encoding noise (FR-001 safe to retire?)
- [ ] Docling eliminates image placeholders (FR-003 safe to retire?)
- [ ] Docling furniture separation removes page numbers / headers (FR-004 safe to retire?)
- [ ] Docling reading order handles multi-column layouts correctly
- [ ] Table quality: Docling structured tables vs pymupdf4llm pipe-table text

### Chunking layer
- [ ] Run B stub count vs Run A-proxy (does Docling Markdown chunk cleaner?)
- [ ] Run C stub count vs Run B (does HybridChunker beat HeadingChunker?)
- [ ] HybridChunker heading metadata coverage (replaces BreadcrumbExtractor?)

### Game-specific rules
- [ ] Stat blocks: intact in Docling output?
- [ ] Frontmatter / copyright pages: handled by Docling or still need our heuristics?

### Performance
- [ ] Docling conversion time acceptable for re-ingestion workflow?

### Recommended 012 spec scope
*(Write the spec framing here based on the findings above)*